# Session XII: Working with APIs and JSON Data

> Requires internet connection. All APIs are free, no auth needed.

---

## Exercise 1: First API Call

Fetch exchange rates from `https://open.er-api.com/v6/latest/USD`. Print the status code, the base currency (the `base_code` field) and 5 rates (inside `rates`). Handle errors.

### How to approach this code

The request demonstrates the read cycle: send GET, inspect status, parse JSON, and display selected rates. A timeout bounds waiting and `raise_for_status()` exposes HTTP failures. The keys used to access the response must match the actual API schema.


In [ ]:
import requests

# TODO: GET request, check status, parse JSON, print 5 rates, try/except

---
## Exercise 2: Currency Converter

Build `convert_currency(amount, from_curr, to_curr)` with error handling and timeout. Get the rates for the source currency from `https://open.er-api.com/v6/latest/{FROM}` (for example `https://open.er-api.com/v6/latest/EUR`).

> This API answers an unknown currency code with status 200 and the body `{"result": "error", "error-type": "unsupported-code"}`, so also check that `result` is `"success"` before reading `rates`.

### How to approach this code

Currency conversion retrieves rates relative to the source currency and multiplies the amount by the target rate. Normalize currency codes and handle missing rates or request failures. If `None` represents failure, callers must check it before using the result in arithmetic or formatting.


In [ ]:
import requests

def convert_currency(amount: float, from_curr: str, to_curr: str) -> float | None:
    # TODO
    pass

print(convert_currency(100, "USD", "EUR"))
print(convert_currency(50, "EUR", "GBP"))
print(convert_currency(100, "USD", "INVALID"))

---
## Exercise 3: Country Data Fetcher

Download the course country dataset from `https://raw.githubusercontent.com/ie-tech-university/programming-with-python-i/main/data/countries.json` (a JSON list in the REST Countries format: `name.common`, `capital`, `population`, `region`, …) and build `fetch_country(name)`. It returns a dict with name, capital, population and region for the country whose `name.common` matches `name` (case-insensitive), or `None` if there is no match. Handle HTTP errors such as a 404 or a timeout.

> The public REST Countries API v3.1 was shut down in 2026 (v5 needs an API key), so the course serves the same data from its repository. Offline? Read `data/countries.json` from your clone instead.

### How to approach this code

The country fetcher converts an API response into a small dictionary for the rest of the program. Search responses may be lists with nested fields, so the code must select a result and extract fields carefully. A nonexistent-country test demonstrates handling a 404 as an expected failure case.


In [ ]:
import requests

def fetch_country(name: str) -> dict | None:
    # TODO
    pass

print(fetch_country("spain"))
print(fetch_country("atlantis"))

---
## Exercise 4: API Caching

Build `cached_fetch(url, cache_file, max_age=3600)` that saves responses to file and reuses them while they are fresh. Store the URL together with the data, so a cache file written for one URL is never returned for a different URL.

### How to approach this code

The cache compares the saved file's age to `max_age`; fresh JSON is reused, while stale or absent data triggers a request and a new save. This reduces network traffic. A reliable implementation distinguishes malformed cache data from a network failure and does not overwrite good cached data after a failed request.


In [ ]:
import json
import os
import time
import requests

def cached_fetch(url: str, cache_file: str, max_age: int = 3600) -> dict | None:
    # TODO
    pass

data = cached_fetch("https://open.er-api.com/v6/latest/EUR", "api_cache.json", 60)
if data: print(f"EUR->USD: {data.get('rates', {}).get('USD')}")
# Second call should use the cache
data2 = cached_fetch("https://open.er-api.com/v6/latest/EUR", "api_cache.json", 60)
# A different URL must not reuse the EUR data
data3 = cached_fetch("https://open.er-api.com/v6/latest/USD", "api_cache.json", 60)
if data3: print(data3.get("base_code"))  # USD

---
## Exercise 5: Multi-Currency Report

**Capstone.** Build `multi_currency_report(transactions)` that groups expenses by original currency and shows EUR totals.

### How to approach this code

Transactions include source-currency values and normalized EUR values. Grouping by source currency is valid, but summing amounts from different currencies together is not; use EUR for a common comparison. This example connects API rates with grouping and report calculations.


In [ ]:
# Given sample data:
transactions = [
    {"amount_eur": 25.0, "original_amount": 27.5, "original_currency": "USD", "category": "Food", "description": "Lunch in NYC"},
    {"amount_eur": 50.0, "original_amount": 50.0, "original_currency": "EUR", "category": "Transport", "description": "Train"},
    {"amount_eur": 42.0, "original_amount": 35.0, "original_currency": "GBP", "category": "Entertainment", "description": "London show"},
]

def multi_currency_report(transactions):
    # TODO: Group by currency, show totals
    pass

multi_currency_report(transactions)

---
## Exercise 6: Cached Currency Lookup
**Objective:** Build a resilient API fetch that caches results and handles failures cleanly.

Write a function `get_exchange_rates(base_currency="USD", cache_file="rates_cache.json")` that:
- fetches `https://open.er-api.com/v6/latest/{base_currency}` with `requests.get()` and a timeout
- handles `Timeout`, `HTTPError`, and general `RequestException`
- treats a response whose `result` is not `"success"` as a failure
- returns cached data when the file is fresh **and** holds rates for the requested `base_currency` (its `base_code`)
- saves the latest JSON response when the fetch succeeds

This exercise simulates the kind of API code used in real dashboards and automation tasks.

### How to approach this code

This version combines cache freshness, JSON read/write, a request timeout, and failure handling. It should return the same data shape whether the result came from cache or network, and write the cache only after a successful response. These choices keep downstream code predictable during outages.


In [ ]:
import json
import os
import time
import requests


def get_exchange_rates(base_currency: str = "USD", cache_file: str = "rates_cache.json") -> dict:
    """Fetch exchange rates with a timeout and a simple cache."""
    # TODO: if cache_file exists, is recent and its "base_code" matches base_currency, return it
    # otherwise call requests.get(..., timeout=10), raise_for_status(), check "result",
    # and save the JSON to cache_file
    # catch request exceptions and return an empty dict on failure
    pass


# Example usage:
# data = get_exchange_rates("USD")
# print(data.get("rates", {}).get("EUR"))
# print(get_exchange_rates("EUR")["base_code"])  # EUR, not the cached USD data